# Regras de Classificação – Algoritmo 1R (One Rule)


In [9]:
import pandas as pd
import numpy as np

url = "https://docs.google.com/spreadsheets/d/1g1aQ61vijh6uHJuc8sijeBEMsoIQ2a5yLwUK04Wptlg/export?format=csv"
df = pd.read_csv(url)
df.head()

,Carimbo de data/hora,Você ficou gripado no ano passado ?,Você tomou vacina da gripe no ano passado?,"Você frequentou no ano passado, semanalmente ambientes com muitas pessoas? (salas cheias, ônibus, eventos, etc.)",Você viajou no ano passado mais de 100 km de distância?,"Você tem alergia nas vias aéreas (rinite, sinusite, etc.)?",Quantas horas você dormiu em média por noite no ano passado?,Você praticou atividade física no ano passado?,Você se alimentou de forma balanceada no ano passado?,"Em média, quantas vezes você lavou as mãos por dia no ano passado?","Na sua percepção, o seu nível de estresse no ano passado foi:"
0,24/03/2026 15:01:35,Sim,Sim,Sim,Poucas vezes por ano,Médio,4 horas ou menos,Sim,Às vezes,3 a 5 vezes,5.0
1,24/03/2026 15:04:20,Sim,Sim,Sim,Nuca,Não,entre 4 e 6 horas,Não,"Não, raramente",Mais de 10 vezes,3.0
2,24/03/2026 15:04:20,Sim,Não,Sim,Poucas vezes por ano,Pouco,mais de 6 horas,Sim,Às vezes,6 a 10 vezes,3.0
3,24/03/2026 15:04:37,Sim,Não,Não,Nuca,Muito,mais de 6 horas,Sim,Às vezes,2 vezes ou menos,2.0
4,24/03/2026 15:05:27,Sim,Sim,Sim,Pelo menos uma vez por mês,Médio,entre 4 e 6 horas,Não,Às vezes,6 a 10 vezes,4.0


In [10]:

mapping = {
    "Carimbo de data/hora": "timestamp",
    "Você ficou gripado no ano passado ?": "gripe",
    "Você tomou vacina da gripe no ano passado?": "vacina",
    "  Você frequentou no ano passado,  semanalmente ambientes com muitas pessoas? (salas cheias, ônibus, eventos, etc.)  ": "ambientes",
    "  Você viajou no ano passado mais de 100 km de distância?  ": "viajou",
    "  Você tem alergia nas vias aéreas (rinite, sinusite, etc.)?  ": "alergia",
    "Quantas horas você dormiu em média por noite no ano passado?": "sono",
    "Você praticou atividade física no ano passado?": "exercicio",
    "Você se alimentou de forma balanceada no ano passado?": "alimentacao",
    "Em média, quantas vezes você lavou as mãos por dia no ano passado?": "lavagem",
    "Na sua percepção, o seu nível de estresse no ano passado foi:": "estresse"
}

df = df.rename(columns=mapping)
df = df.drop(columns=['timestamp'], errors='ignore')

for col in df.columns:
    df[col] = df[col].astype(str).str.strip()

df = df.replace({'nan': np.nan})
df = df.fillna('desconhecido')

df.head()

,gripe,vacina,ambientes,viajou,alergia,sono,exercicio,alimentacao,lavagem,estresse
0,Sim,Sim,Sim,Poucas vezes por ano,Médio,4 horas ou menos,Sim,Às vezes,3 a 5 vezes,5.0
1,Sim,Sim,Sim,Nuca,Não,entre 4 e 6 horas,Não,"Não, raramente",Mais de 10 vezes,3.0
2,Sim,Não,Sim,Poucas vezes por ano,Pouco,mais de 6 horas,Sim,Às vezes,6 a 10 vezes,3.0
3,Sim,Não,Não,Nuca,Muito,mais de 6 horas,Sim,Às vezes,2 vezes ou menos,2.0
4,Sim,Sim,Sim,Pelo menos uma vez por mês,Médio,entre 4 e 6 horas,Não,Às vezes,6 a 10 vezes,4.0


## Implementação do Algoritmo 1R

In [11]:
def train_one_rule(df, target):
    features = [c for c in df.columns if c != target]
    resultados = []

    for atributo in features:
        erro_total = 0
        regras = {}

        for valor in df[atributo].unique():
            subset = df[df[atributo] == valor]
            contagem = subset[target].value_counts()


            if contagem.empty:
                classe_majoritaria = 'desconhecido'
                acertos = 0
                erros = len(subset)
            else:
                classe_majoritaria = contagem.idxmax()
                acertos = contagem.max()
                erros = len(subset) - acertos

            regras[valor] = {
                'classe': classe_majoritaria,
                'erros': erros,
                'total': len(subset)
            }
            erro_total += erros

        taxa_erro = erro_total / len(df)

        resultados.append({
            'atributo': atributo,
            'regras': regras,
            'erro_total': erro_total,
            'taxa_erro': taxa_erro
        })

    resultados = sorted(resultados, key=lambda x: x['taxa_erro'])
    return resultados[0]

def predict_one_rule(df, best_attribute, rules_dict):
    predictions = []
    for index, row in df.iterrows():
        value = row[best_attribute]
        if value in rules_dict:
            predictions.append(rules_dict[value]['classe'])
        else:

            predictions.append('UNKNOWN')
    return pd.Series(predictions, index=df.index)

target = 'gripe'

melhor_modelo = train_one_rule(df.copy(), target)

print('Melhor atributo:', melhor_modelo['atributo'])
print('Taxa de erro:', melhor_modelo['taxa_erro'])

Melhor atributo: viajou
Taxa de erro: 0.3655913978494624


## Regras Geradas pelo 1R

In [12]:
print('Regras Geradas pelo 1R:')
atributo = melhor_modelo['atributo']
for valor, info in melhor_modelo['regras'].items():
    print(f"SE {atributo} = {valor} ENTÃO gripe = {info['classe']} (erros {info['erros']}/{info['total']})")

y_pred = predict_one_rule(df, melhor_modelo['atributo'], melhor_modelo['regras'])
y_true = df[target]


accuracy = (y_pred == y_true).sum() / len(y_true)
print(f"\nAcurácia do modelo 1R no conjunto de treinamento: {accuracy:.2f}")

Regras Geradas pelo 1R:
SE viajou = Poucas vezes por ano ENTÃO gripe = Sim (erros 43/121)
SE viajou = Nuca ENTÃO gripe = Sim (erros 5/16)
SE viajou = Pelo menos uma vez por mês ENTÃO gripe = Não (erros 20/49)

Acurácia do modelo 1R no conjunto de treinamento: 0.63
